# 拓展：漏斗分析深度拓展——从点估计到不确定性量化与自动化工程

> **本拓展新增 7 个主课程未覆盖的深度概念**
> 建议学习时间：2-2.5小时 | 必修概念：5个 | 选修概念：2个

---

## 知识网络预览

```
                     ┌──────────────────────────────────────┐
                     │      主课核心：漏斗分析               │
                     │   (点估计 + 分群 + 微转化 + 逆向)     │
                     └────────────┬─────────────────────────┘
                                  │
          ┌───────────────────────┼───────────────────────┐
          │                       │                       │
    ┌─────▼─────┐          ┌─────▼─────┐          ┌─────▼─────┐
    │ [前置]    │          │ [并行]    │          │ [防护]    │
    │ 置信区间  │          │ 辛普森    │          │ 异常检测  │
    │ 贝叶斯漏斗│          │ 悖论陷阱  │          │ 博弈防护  │
    └─────┬─────┘          └─────┬─────┘          └─────┬─────┘
          │                       │                       │
          └───────────────────────┼───────────────────────┘
                                  │
                     ┌────────────┴───────────┐
                     │                        │
               ┌─────▼─────┐            ┌─────▼─────┐
               │ [落地]    │            │ [替代]    │
               │ 监控看板  │            │ HEART框架 │
               └───────────┘            └───────────┘
```

**阅读指引**：
- 如果你主课练习全部完成 → 直接从入口自诊开始
- 如果你对「转化率只是一个数字，它的可靠性如何」有疑问 → 先看概念1+2（置信区间+贝叶斯）
- 如果你曾被「整体提升但分群都在跌」困惑 → 先看概念3（辛普森悖论）
- 如果你想搭建自动化的漏斗监控 → 重点看概念4+7

**概念清单**：

| # | 概念 | 深度类型 | 标记 | 关系类型 |
|---|------|---------|------|---------|
| 1 | 漏斗转化率的置信区间与二项标准误 | 数学+代码 | [必修] | [前置] |
| 2 | 贝叶斯漏斗：Beta-Binomial后验更新 | 数学+代码 | [必修] | [前置] |
| 3 | 辛普森悖论的形式化陷阱与检测器 | 数学+案例 | [必修] | [并行] |
| 4 | 漏斗异常模式自动检测器 | 代码 | [必修] | [防护] |
| 5 | 漏斗指标的博弈与古德哈特定律 | 博弈+代码 | [选修] | [防护] |
| 6 | HEART框架：Google UX视角的转化衡量 | 案例 | [选修] | [替代] |
| 7 | 漏斗监控看板的工程化设计 | 工程+代码 | [必修] | [落地] |

## 第一部分：沙盘设定

### 🏢 公司画像：悦选（YueXuan）

**产品**：中高端女性时尚电商平台，连接独立设计师与消费者。主打「设计师品牌 + 内容种草」模式——用户在App内浏览设计师故事、穿搭Lookbook、真人试穿测评后下单。

**商业模式**：平台抽佣制（佣金率15%-25%），核心公式：GMV = 流量 x 转化率 x 客单价。当前阶段：B轮融资后18个月，团队200人，MAU约50万，月GMV约1800万元。

**核心数据**：
- 获客成本(CAC)过去6个月从45元涨到72元（+60%），主要靠信息流投放拉新
- 整体支付转化率（打开App→支付成功）连续3个月停滞在6.0%-6.3%
- 同期MAU增长了40%（50万 vs 年初36万）——流量涨了，转化率没涨

**CEO的灵魂拷问**：「我们花了更多钱买流量，但GMV不成比例增长——到底是漏斗坏了，还是我们在买垃圾流量？」

### 🎯 当前挑战与你的角色

**你的角色**：悦选高级数据分析师，带领2人分析小组，直接汇报给增长VP。

**本季度三大挑战**：
1. **转化率停滞**：支付转化率6%已持续3个月——CEO要求「两周内给出根因分析和优化方案」
2. **渠道ROI恶化**：信息流渠道CAC=72元但该渠道用户7日留存只有12%（自然流量是35%）——增长团队内部在争论「继续买量」还是「先修漏斗」
3. **数据基础设施薄弱**：漏斗数据靠分析师每周手动跑SQL→Excel→PPT，没有实时监控。上次支付接口故障过了6小时才被发现

**你的KPI**：
- 两周内定位转化率停滞的根因（验收标准：CEO采纳你的分析结论并启动一个优化项目）
- 一个月内搭建漏斗自动化监控看板（验收标准：异常发生后15分钟内告警）

### 💾 数据资产

悦选的数据仓库遵循四层分层架构（ODS→DWD→DWS→ADS），你可以访问以下表：

| 表名 | 层级 | 关键字段 | 更新频率 |
|------|------|---------|---------|
| `ods.user_event_log` | ODS | user_id, event_name, session_id, page_url, device_type, channel, timestamp, properties(JSON) | 实时 |
| `dwd.user_behavior_facts` | DWD | user_id, dt, event_name, prev_event, next_event, session_duration, page_stay_sec | T+1 |
| `dwd.order_facts` | DWD | order_id, user_id, order_status, total_amount, payment_method, product_ids, created_at | 实时 |
| `dim.users` | DIM | user_id, registration_dt, channel_source, device_type, city_tier, user_segment | T+1 |
| `dim.products` | DIM | product_id, category, designer_id, price_range, avg_rating, review_count | T+1 |
| `dwd.cart_events` | DWD | user_id, product_id, event_type, session_id, timestamp | 实时 |
| `dws.daily_funnel_agg` | DWS | dt, funnel_step, channel, device_type, user_segment, user_count, total_session_count | T+1 |
| `ads.channel_performance` | ADS | dt, channel, spend, impressions, clicks, new_users, 7d_ltv, 30d_ltv, roi | T+1 |

> ⚠️ 注意：`dws.daily_funnel_agg` 是预聚合表（按日×漏斗步骤×渠道×设备×用户分群的组合维度聚合），查询快但灵活性有限。做深度分析时需要回退到 DWD 层自己算。

### 👥 利益相关方地图

| 角色 | 人名 | 关切 | 数据素养 | 沟通偏好 |
|------|------|------|---------|---------|
| CEO | 张总 | GMV、ROI、市场份额 | 中（会看图表不懂统计） | 一页纸摘要，结论先行 |
| 增长VP(你的老板) | 李总 | 转化率、渠道效率、团队KPI | 高（前麦肯锡顾问） | 结构化分析报告 |
| 市场投放负责人 | 王哥 | CAC、渠道ROI、预算分配 | 中 | 看ROI数字，容易陷入「再投一周试试」 |
| 产品经理(转化组) | 赵姐 | 加购率、支付成功率、UX | 中高（做过AB测试） | 想要「精确到页面的问题定位」 |
| 后端工程师 | 陈工 | 系统稳定性、接口响应时间 | 低(数据领域) | 需要「哪张表哪个字段出了什么问题」 |

---
## 第二部分：入口自诊

> 以下问题**只测主课程内容**。答不上来就先回主课复习，不要硬闯拓展。

### 自诊题1：识别漏斗形状

In [ ]:
# 自诊题1：以下是某SaaS产品的注册漏斗，判断漏斗形状类型
import numpy as np

steps = ['访问官网', '点击注册', '输入手机号', '输入验证码', '设置密码', '注册成功']
counts = np.array([10000, 8000, 6400, 1600, 1280, 1152])

step_conv = counts[1:] / counts[:-1]
step_loss = counts[:-1] - counts[1:]

print('步骤转化率:', [f'{c:.1%}' for c in step_conv])
print('流失人数:', [f'{l:,}' for l in step_loss])

# TODO: 判断漏斗形状类型并给出诊断方向
# 你的答案：
#   形状类型: _______
#   诊断方向: _______

In [ ]:
# 自诊题2：构造"整体掩盖问题"的数据案例
# 使得整体加购率=35%（看起来还行），但新用户只有15%
import numpy as np

new_browse = 2400; old_browse = 2600
new_cart_rate = 0.15; overall_cart_rate = 0.35

new_cart = int(new_browse * new_cart_rate)
old_cart = int(overall_cart_rate * (new_browse + old_browse) - new_cart)

print(f"新用户加购数: {new_cart} (加购率 {new_cart/new_browse:.0%})")
print(f"老用户加购数: {old_cart} (加购率 {old_cart/old_browse:.0%})")
print(f"整体加购率: {(new_cart+old_cart)/(new_browse+old_browse):.0%}")

In [ ]:
# 自诊题3：转化窗口判断
# 某电商产品，用户决策周期约3-5天（比价+凑单），DAU约10万
# (1) 哪个窗口最可能低估转化率？为什么？
# (2) 哪个窗口可能引入"偶然关联"的风险？
# (3) 你推荐什么窗口？

print("(1) 最可能低估转化率的窗口: ___")
print("   理由: ___")
print("(2) 可能引入偶然关联的窗口: ___")
print("   理由: ___")
print("(3) 推荐窗口: ___")
print("   理由: ___")

### 自诊反馈

<details>
<summary><b>点击展开：答案 + 常见偏差 + 补救路径</b></summary>

**题1答案**：
- 形状类型：**中间断崖**。前两步转化率正常（80%），但「手机号→验证码」从6400跌到1600（转化率仅25%）
- ❌ 常见偏差①：判断为「逐渐收窄」——端到端11.5%在SaaS不算差。错误：没逐步骤检查转化率
- ❌ 常见偏差②：判断为「最后一步暴跌」——底部的1152最小。错误：最后一步转化率90%，根本不差
- 🔧 补救：重读主课2.2节「漏斗的常见形状及其诊断」

**题2答案**：
- new_cart = 2400 x 0.15 = 360; old_cart = 0.35 x 5000 - 360 = 1390; 老用户加购率 = 53.5%
- ❌ 常见偏差①：设new_cart=600（把15%算成了25%）
- ❌ 常见偏差②：直觉认为「整体35%≈新老都是35%左右」——这正是整体掩盖问题的心理根源
- 🔧 补救：手算一遍不要跳步

**题3答案**：
- (1) 1天窗口最可能低估——决策周期3-5天，多数人第一天还没完成决策
- (2) 30天窗口——30天后回来买可能跟30天前的浏览无关
- (3) 推荐7天窗口——决策周期的1.5-2倍
- ❌ 常见偏差①：选3天。错误：选下限仍会低估
- ❌ 常见偏差②：选30天。错误：忽视了偶然关联和时效性
- 🔧 补救：重读主课2.4节和2.8节

</details>

---
## 第三部分：概念网络展开

---

### 概念1：漏斗转化率的置信区间与二项标准误 [必修] [前置]

> **关系类型**：[前置]——主课教你算转化率（点估计），这个概念教你怎么判断转化率是否可靠
> **来自空白区**：Bootstrap CI + 统计推断基础（永久空白区3.1节）

#### 🔴 认知冲突

假设你负责悦选的漏斗周报。上周「浏览→加购」转化率38%，这周35%——跌了3pp。

产品经理说：「跌了3个点！加购流程肯定出bug了！」
你说：「等一下……我先看看这个35%是基于多少样本算的。」
产品经理：「多少样本有区别吗？35%就是35%。」

**问题**：如果上周38%基于10000次浏览，这周35%基于100次浏览——你的结论会一样吗？

<details><summary>思考后展开</summary>
n=100时35人加购，真实转化率可能在26%-45%（95%CI）。与上周38%（基于10000样本，CI[37%,39%]）相比——CI有重叠，不能断定真跌了。
**核心insight**：主课教你看转化率的「值」，但没教你看「可靠性」。当用小样本做分群漏斗时，转化率波动可能只是噪声。
</details>

#### 概念引入

**① 一句话白话**：转化率不是一个精确数字——它是基于样本的估计。「抛10次硬币5次正面=50%」和「抛10000次5000次正面=50%」——数字一样，但你对前者的信心远低于后者。

**② 核心逻辑 + 具体例子**：

漏斗每步转化是**二项过程**：用户到达→要么转化(1)要么不转化(0)。用 p = 转化人数/到达人数 估计真实转化概率。

**二项分布标准误**：SE = sqrt(p(1-p)/n)，其中n是到达步骤的人数。
**95%置信区间（正态近似）**：p ± 1.96 × SE

**悦选实例**：

| 场景 | 转化率 | n | SE | 95% CI |
|------|--------|----|-----|--------|
| 上周(大样本) | 38% | 10,000 | 0.49% | [37.1%, 39.0%] |
| 这周(小样本) | 35% | 100 | 4.77% | [25.7%, 44.3%] |

CI有重叠 → 不能断定转化率真跌了。小样本35%可能只是因为35人加购——多1个人就是36%。

**③ 与核心概念的关系**：
- 关系类型：[前置]——判断漏斗变化是否显著的统计基础
- 主课教你「最大漏水点是浏览→加购（40%）」，但没教怎么判断40%是稳定值还是随机波动
- W7-W8会深入讲置信区间和假设检验，在漏斗场景下你现在就能直接用

#### 在沙盘中的用武之地

悦选CEO问：「支付转化率从6.2%跌到6.0%——漏斗坏了还是正常波动？」

预聚合表 `dws.daily_funnel_agg` 只有人数和转化率，没有置信区间。你需要从DWD层自己算：

| 日 | 支付人数 | DAU | 转化率 | SE | 95% CI |
|----|---------|-----|--------|-----|--------|
| 昨天 | 12,400 | 200,000 | 6.20% | 0.054% | [6.09%, 6.31%] |
| 今天 | 12,000 | 200,000 | 6.00% | 0.053% | [5.89%, 6.11%] |

两个CI有重叠 → **不能断定转化率真跌了**。CEO的「漏斗坏了」直觉可能是错的——数据驱动决策的价值就在于此。

In [ ]:
# 练习1：计算悦选漏斗每一步的置信区间
import numpy as np
from scipy import stats

funnel_steps = ["打开App", "浏览商品", "点击详情", "加入购物车", "进入下单", "支付成功"]
funnel_counts = np.array([50000, 28000, 19500, 8200, 4500, 3000])

def funnel_confidence_intervals(counts, confidence=0.95):
    """为漏斗每一步计算转化率及95%置信区间"""
    z = stats.norm.ppf(1 - (1 - confidence) / 2)
    results = []
    for i in range(1, len(counts)):
        n_arrive = counts[i-1]
        n_convert = counts[i]
        p = n_convert / n_arrive
        se = np.sqrt(p * (1 - p) / n_arrive)
        ci_lower = max(0, p - z * se)
        ci_upper = min(1, p + z * se)
        results.append({"step": f"{funnel_steps[i-1]}->{funnel_steps[i]}",
                        "n_arrive": n_arrive, "conv_rate": p,
                        "se": se, "ci_lower": ci_lower, "ci_upper": ci_upper})
    return results

results = funnel_confidence_intervals(funnel_counts)
print(f"{"步骤":<30} {"到达数":>8} {"转化率":>8} {"SE":>8} {"95% CI":>20}")
print("-" * 80)
for r in results:
    print(f"{r["step"]:<30} {r["n_arrive"]:>8,} {r["conv_rate"]:>7.1%} "
          f"{r["se"]:>7.3%} [{r["ci_lower"]:.1%}, {r["ci_upper"]:.1%}]")

# 问题：如果最后一步只有450人到达（而非4500），SE会变成多少？
n_small = 450
p_last = 3000/4500
se_small = np.sqrt(p_last * (1 - p_last) / n_small)
print(f"\n如果末步到达人数只有{n_small}人:")
print(f"  SE = {se_small:.3%} (原来 SE = {results[-1]["se"]:.3%})")
# 预期：SE从~0.7%膨胀到~2.2%，约3倍（sqrt(10)≈3.16）

#### 反馈

| 维度 | 内容 |
|------|------|
| ✅ 参考答案 | 末步SE最大（到达人数最少）。SE与sqrt(n)成反比——n从4500降到450，SE翻约3倍 |
| ❓ 常见偏差① | 以为「转化率低的步骤SE大」——SE=sqrt(p(1-p)/n)，p接近0.5时分子最大，但n通常主导 |
| ❓ 常见偏差② | CI有重叠就断定「没区别」——不精确。CI轻微重叠时两样本z检验可能仍显著（z = (p1-p2)/sqrt(p_pool(1-p_pool)(1/n1+1/n2)) |
| 🔧 补救① | 手算n=10000(p=0.1) vs n=100(p=0.5)的SE——验证n主导SE |
| 🔧 补救② | 学习两样本比例z检验，理解CI重叠≠不显著 |

---
### 概念2：贝叶斯漏斗——Beta-Binomial后验更新 [必修] [前置]

> **关系类型**：[前置]——置信区间告诉你估计的精度，贝叶斯漏斗让你在样本量不足时借用先验知识
> **来自空白区**：贝叶斯假设更新（永久空白区3.1节「贝叶斯假设更新：似然比 x 先验 = 后验」）

#### 🔴 认知冲突

悦选新上线了「设计师直播」功能。第一周，直播间→加购转化率60%（30人浏览，18人加购）。产品经理兴奋：「60%！比大盘40%高一半！应该把所有流量导向直播间！」

你皱了皱眉：「30人浏览……下周可能就变回12人加购了。这个60%有多稳？」

**问题**：如果悦选大盘历史加购率是40%（基于100万次浏览），你刚看到的60%（基于30次浏览）——你应该信60%还是40%？还是介于两者之间？

#### 概念引入

**① 一句话白话**：当样本很小时，你不该完全信新数据（60%），也不该完全信历史（40%），而是把两者数学地结合起来——贝叶斯就是做这件事的。

**② 核心逻辑**：
- **先验**（prior）：看到新数据前对转化率的已有认知。用 Beta(α, β) 表示，α=历史成功次数+1，β=历史失败次数+1
- **似然**（likelihood）：新观察到的数据。n次试验中k次成功
- **后验**（posterior）：Beta(α+k, β+(n-k))。后验均值 = (α+k)/(α+β+n)

**悦选实例**：
- 先验：大盘加购率40%，相当于 Beta(400, 600)——等效于看过400次成功+600次失败
- 新数据：直播间30人，18人加购
- 后验：Beta(400+18, 600+12) = Beta(418, 612)
- 后验均值 = 418/(418+612) = 40.6%——非常接近历史40%，而非观测到的60%！

这就是贝叶斯的**缩水效应**——小样本信息被先验拉回去了。当n很大时，后验≈似然；当n很小时，后验≈先验。

**③ 与核心概念的关系**：为置信区间补充了贝叶斯视角，在小样本/需持续更新的场景下更自然。

#### 在沙盘中的用武之地

悦选做「分渠道漏斗」时：信息流渠道日均5000人浏览（大样本，可信）；KOL渠道日均仅80人浏览（小样本，转化率忽高忽低）。王哥说KOL转化率最高应该加预算——但贝叶斯缩水后KOL只有40.5%，跟自然流量42%差不多，不值得特殊加预算。你成功阻止了一次错误的资源错配。

In [ ]:
# 练习2：贝叶斯漏斗——Beta-Binomial后验更新
import numpy as np
from scipy import stats

# 悦选大盘历史：浏览->加购转化率40%（基于大量数据）
# 等效先验：Beta(alpha=400, beta=600)，均值=40%
alpha_prior, beta_prior = 400, 600

channels = {
    "信息流(大样本)": {"views": 5000, "carts": 1750},   # 35%
    "KOL(小样本)":    {"views": 80,   "carts": 44},     # 55%
    "自然流量(大样本)": {"views": 12000,"carts": 5040},  # 42%
}

print("贝叶斯缩水效应：小样本观测被先验拉回")
print("=" * 70)
print(f"先验均值: {alpha_prior/(alpha_prior+beta_prior):.1%}")
print(f"{"渠道":<20} {"观测转化率":>10} {"后验均值":>10} {"缩水":>8} {"后验95%可信区间":>22}")
print("-" * 70)

for name, data in channels.items():
    n, k = data["views"], data["carts"]
    observed = k / n
    alpha_post = alpha_prior + k
    beta_post = beta_prior + (n - k)
    posterior_mean = alpha_post / (alpha_post + beta_post)
    ci_lower = stats.beta.ppf(0.025, alpha_post, beta_post)
    ci_upper = stats.beta.ppf(0.975, alpha_post, beta_post)
    shrinkage = (observed - posterior_mean) / observed
    print(f"{name:<20} {observed:>9.1%} {posterior_mean:>10.2%} {shrinkage:>7.1%}  [{ci_lower:.1%}, {ci_upper:.1%}]")

print("\n观察：KOL(小样本)后验被大幅拉回（缩水26%）；大样本渠道后验≈观测值")
print("启示：小渠道不看贝叶斯缩水，很可能把噪声当信号")

#### 反馈

| 维度 | 内容 |
|------|------|
| ✅ 参考答案 | KOL渠道观测55%但后验只有40.7%（缩水26%）。信息流和自然流量n大，后验≈观测值 |
| ❓ 常见偏差① | 以为贝叶斯=主观偏见——先验基于历史数据，是经验证据的数学编码，不是瞎猜 |
| ❓ 常见偏差② | 以为先验越强越好——太强的先验让数据几乎无法改变后验，新渠道的真实改进永远看不到 |
| ❓ 常见偏差③ | 混淆95%CI和95%可信区间——频率学派CI说「重复实验100次，95个CI包含真值」；贝叶斯可信区间说「真值有95%概率落在这个区间」——后者更直觉但前提是先验正确 |
| 🔧 补救① | 修改α+β=100（弱先验）vs α+β=10000（强先验），观察KOL后验变化 |
| 🔧 补救② | 模拟：如果KOL连续4周n=80且转化率都=55%，后验会怎么变？ |

---
### 概念3：辛普森悖论在漏斗中的形式化陷阱与检测器 [必修] [并行]

> **关系类型**：[并行]——主课说了「整体会骗人」但没有给出辛普森悖论的数学结构
> **来自空白区**：辛普森悖论在同期群中的陷阱（技能深度概念速查表）+ 数学形式化

#### 🔴 认知冲突

悦选周报：本周整体「浏览→加购」转化率从上周35%提升到38%——涨了3pp！
但按新老用户拆开：新用户从25%跌到22%；老用户从55%跌到52%——都在跌！

**问题**：两个分群都在跌，整体却涨了——这怎么可能？

#### 概念引入

**① 一句话白话**：辛普森悖论就是「分组看都在跌，合起来看却在涨」——因为各组的人数占比发生了变化。

**② 形式化证明**：总转化率 P = w₁p₁ + w₂p₂

当 p₁'<p₁ 且 p₂'<p₂ 但 P'>P 时，必然 w₁'<w₁ 且 w₂'>w₂——高转化率群体的占比增加了。

**悦选实例**：
| 用户群 | 上周浏览占比 | 上周加购率 | 本周浏览占比 | 本周加购率 |
|--------|-----------|----------|-----------|----------|
| 新用户 | 60% | 25% | 40% | 22% |
| 老用户 | 40% | 55% | 60% | 52% |

上周：0.60×25%+0.40×55%=37%。本周：0.40×22%+0.60×52%=40%。两个群各跌3pp，整体却涨了3pp——老用户占比从40%跳到60%。

**③ 与核心概念的关系**：主课2.3节定性说了「整体会骗人」，这里给出数学证明。知道了占比变化机制才能在看到「整体涨」时主动检查分群。

#### 在沙盘中的用武之地

悦选市场部报告：「信息流渠道加购率从28%提升到31%——投放见效了！」
你分群后发现：iOS用户从40%跌到38%，Android从10%跌到8%——都在跌！但因为Android占比从40%涨到60%（信息流在安卓端扩量），整体看起来涨了。你判断：投放优化没有真提升转化——是用户结构变了。

In [ ]:
# 练习3：辛普森悖论检测器
import numpy as np

def simpson_check(before_groups, after_groups):
    """
    检测辛普森悖论——分群都在同方向变化但整体反方向变化
    before_groups/after_groups: list of (count, rate) tuples
    """
    before_total = sum(c for c, _ in before_groups)
    after_total = sum(c for c, _ in after_groups)
    before_overall = sum(c * r for c, r in before_groups) / before_total
    after_overall = sum(c * r for c, r in after_groups) / after_total

    group_dirs = []
    for (cb, rb), (ca, ra) in zip(before_groups, after_groups):
        if ra > rb: group_dirs.append("up")
        elif ra < rb: group_dirs.append("down")
        else: group_dirs.append("flat")

    all_down = all(d == "down" for d in group_dirs)
    all_up = all(d == "up" for d in group_dirs)
    overall_up = after_overall > before_overall

    paradox = (all_down and overall_up) or (all_up and not overall_up)
    return {"paradox_detected": paradox, "before_overall": before_overall,
            "after_overall": after_overall, "group_directions": group_dirs}

# 测试：悦选案例
before = [(6000, 0.25), (4000, 0.55)]
after  = [(4000, 0.22), (6000, 0.52)]
result = simpson_check(before, after)
print(f"检测到悖论: {result["paradox_detected"]}")
print(f"整体: {result["before_overall"]:.1%} -> {result["after_overall"]:.1%}")
print(f"各群方向: {result["group_directions"]}")

# 挑战：构造反向悖论（分群涨但整体跌）
print("\n构造反向辛普森悖论...")
cb = [(8000, 0.30), (2000, 0.60)]  # 高转化群占比小
ca = [(9500, 0.31), (500, 0.61)]   # 高转化群占比更小
cr = simpson_check(cb, ca)
print(f"悖论: {cr["paradox_detected"]}, 整体方向: {"up" if cr["after_overall"]>cr["before_overall"] else "down"}")

#### 反馈

| 维度 | 内容 |
|------|------|
| ✅ 参考答案 | 反向悖论：高转化群体从20%占比暴跌到5%，即使两个群转化率都在涨，整体被低转化群"稀释"到下跌 |
| ❓ 常见偏差① | 看到分群同方向就以为整体一定同方向——加权平均=权重×值，权重变化可能逆转值的效应 |
| ❓ 常见偏差② | 修复方法：标准化占比不变，报告「如果用户结构不变，转化率实际变化是X」 |
| 🔧 补救 | 每次漏斗对比，强制输出三列：①整体变化 ②各分群变化 ③各分群占比变化。悖论现形 |

---
### 概念4：漏斗异常模式自动检测器 [必修] [防护]

> **关系类型**：[防护]——主课教你手动看漏斗形状判断问题，这个概念让你写代码自动检测，防止「看漏了」
> **来自空白区**：异常检测算法（永久空白区3.1节）+ 虚荣指标自动检测器思路

#### 🔴 认知冲突

你是悦选唯一的分析师。每天早上一到公司就要看前一天的漏斗数据——6步骤×4渠道×2设备=至少48个数字。你能每天稳定发现异常吗？

上周三，「银行卡支付→支付成功」转化率从85%跌到65%——但你那天忙着做CEO的PPT，只扫了一眼数据没注意到。等发现时过了2天，约300个用户尝试银行卡支付失败——损失约5.4万元。

**问题**：靠人眼每天检查漏斗异常——可靠吗？如果不靠人，应该靠什么？

#### 概念引入

**① 一句话白话**：给漏斗装一个「自动报警器」——就像烟雾报警器不会等房子烧起来才响，检测到烟雾就立刻响。

**② 核心逻辑**：自动化异常检测需要三样东西：基线（正常范围）、检测规则（什么算异常）、告警（通知谁）。

**六种漏斗异常模式**：第一步断崖、中间断崖、最后一步暴跌、倒漏斗、趋势漂移、置信区间突破（当前值超出历史基线95%CI）

#### 在沙盘中的用武之地

悦选目前漏斗异常靠「分析师肉眼扫+有人投诉才排查」。上次支付接口挂了6小时才被发现——对B轮公司，6小时可能意味着数十万元直接损失。需要构建 FunnelMonitor 类，每天早上自动检测前一天漏斗数据，发现异常自动推送消息。

In [ ]:
# 练习4：漏斗异常模式自动检测器
import numpy as np
from scipy import stats
from dataclasses import dataclass
from typing import List

@dataclass
class FunnelAlert:
    step_index: int
    alert_type: str
    severity: str  # critical/warning/info
    message: str
    current_value: float
    baseline_value: float

class FunnelMonitor:
    """漏斗异常自动检测器"""
    def __init__(self, step_names, first_step_threshold=0.20,
                 mid_cliff_ratio=0.50, last_step_threshold=0.50, trend_days=5):
        self.step_names = step_names
        self.first_step_threshold = first_step_threshold
        self.mid_cliff_ratio = mid_cliff_ratio
        self.last_step_threshold = last_step_threshold
        self.trend_days = trend_days
        self.history = []

    def add_observation(self, counts, date):
        convs = counts[1:] / counts[:-1]
        self.history.append((counts, convs, date))
        if len(self.history) > 60:
            self.history.pop(0)

    def check(self, counts):
        convs = counts[1:] / counts[:-1]
        alerts = []
        # 1. 倒漏斗检测
        for i in range(1, len(counts)):
            if counts[i] > counts[i-1]:
                alerts.append(FunnelAlert(i, "reverse", "critical",
                    f"倒漏斗: {self.step_names[i-1]}->{self.step_names[i]}",
                    float(counts[i]), float(counts[i-1])))
        # 2. 第一步断崖
        if convs[0] < self.first_step_threshold:
            alerts.append(FunnelAlert(0, "first_step_cliff", "warning",
                f"第一步转化率仅{convs[0]:.1%}", convs[0], self.first_step_threshold))
        # 3. 中间断崖
        for i in range(1, len(convs)-1):
            if convs[i-1] > 0.3 and convs[i] / max(convs[i-1], 0.01) < self.mid_cliff_ratio:
                alerts.append(FunnelAlert(i, "mid_cliff", "warning",
                    f"中间断崖: {convs[i-1]:.1%}->{convs[i]:.1%}", convs[i], convs[i-1]))
        # 4. 最后一步暴跌
        if all(c > 0.5 for c in convs[:-1]) and convs[-1] < self.last_step_threshold:
            alerts.append(FunnelAlert(len(convs)-1, "last_step_crash", "critical",
                f"最后一步暴跌: {convs[-1]:.1%}", convs[-1], self.last_step_threshold))
        # 5. CI突破检测
        if len(self.history) >= 14:
            for i in range(len(convs)):
                hist_convs = [h[1][i] for h in self.history[-30:]]
                mean_c, std_c = np.mean(hist_convs), np.std(hist_convs)
                if std_c > 0 and abs(convs[i] - mean_c) > 3 * std_c:
                    alerts.append(FunnelAlert(i, "ci_break", "critical",
                        f"3-sigma突破: 当前{convs[i]:.1%} vs 基线{mean_c:.1%}+-{3*std_c:.2%}",
                        convs[i], mean_c))
        return alerts

# 测试
step_names = ["打开App", "浏览商品", "点击详情", "加入购物车", "进入下单", "支付成功"]
monitor = FunnelMonitor(step_names)

# 喂30天正常数据建立基线
np.random.seed(42)
for day in range(30):
    base = np.array([50000, 28000, 19500, 8200, 4500, 3000])
    noise = np.random.normal(0, 200, len(base)).astype(int)
    normal = np.maximum(base + noise, 1)
    for i in range(1, len(normal)):
        normal[i] = min(normal[i], normal[i-1] - 1)
    monitor.add_observation(normal, f"day_{day}")

# 检测异常
bad = np.array([50000, 28000, 19500, 19500, 4500, 900])  # 倒漏斗+末步暴跌
alerts = monitor.check(bad)
print(f"检测到 {len(alerts)} 个异常:")
for a in alerts:
    print(f"  [{a.severity.upper()}] {a.alert_type}: {a.message}")

#### 反馈

| 维度 | 内容 |
|------|------|
| ✅ 参考答案 | 倒漏斗（加购=点击详情）和最后一步暴跌（下单→支付20%<50%）都会被检测到 |
| ❓ 常见偏差① | 以为检测器能替代分析判断——检测器是看门狗不是侦探。它告诉你有异常，不告诉你为什么和怎么办 |
| ❓ 常见偏差② | 阈值太敏感→告警疲劳；太迟钝→漏报。需要根据业务容忍度调参并定期review |
| 🔧 补救 | 增加「告警确认」机制：检测到异常后自动跑分群维度诊断，把「哪个子群出问题」一并推送 |

---
### 概念5：漏斗指标的博弈与古德哈特定律 [选修] [防护]

> **关系类型**：[防护]——前面概念帮你精确测量和自动检测，但还有一个更根本的问题：你测的东西本身可能被污染
> **来自空白区**：指标腐烂生命周期 + 虚荣指标自动检测器（永久空白区5.3节）

#### 🔴 认知冲突

悦选市场团队KPI是「新用户注册数」。上个月超额完成120%——CEO很高兴。
但你看了新注册用户的漏斗数据：注册→浏览35%（正常55%）、浏览→加购8%（正常25%）、7日留存6%（正常20%）。
市场团队最近在投放「注册送20元券」的信息流广告——素材强调「注册即送钱」，而非「这里有好看的衣服」。

**问题**：市场团队的KPI超额完成了——但这对公司是好事还是坏事？

#### 概念引入

**① 一句话白话**：古德哈特定律——「当一个指标成为目标，它就不再是一个好指标。」因为人会针对这个指标优化，有时以牺牲真正的用户价值为代价。

**② 漏斗指标的六种博弈模式**：
| 博弈模式 | 具体表现 | 检测方法 |
|---------|---------|---------|
| 补贴污染 | 用现金/优惠券诱导用户完成某步骤 | 补贴组 vs 无补贴组的后续留存对比 |
| 口径操作 | 修改指标定义使数字好看 | 定期审计指标定义和埋点逻辑 |
| 筛选偏差 | 只向高转化概率用户展示某功能 | 比较「看到功能」vs「全体用户」特征分布 |
| 鸡血流量 | 投激励视频广告获取「为拿奖励」的用户 | 按渠道拆留存——激励渠道次日留存通常<5% |
| 用户疲劳 | 弹窗/推送过度催促，短期转化涨长期留存跌 | 转化率趋势 vs 30日留存趋势——背离告警 |
| 虚假繁荣 | 机器人/刷量 | 行为模式异常检测（间隔时间精确一致） |

**博弈诊断三步法**：
1. 检查KPI上升是否伴随反指标下降（注册↑+留存↓=补贴污染）
2. 按「是否被补贴/激励」分群对比后续留存
3. 为KPI设置护栏指标（如：新用户30日LTV > CAC）

**③ 与核心概念的关系**：漏斗分析告诉你哪里漏水，但你要先确保没人在往漏斗里注水。从「这个指标是多少」到「这个指标还能被信任吗」。

#### 在沙盘中的用武之地

你向CEO汇报：「市场团队完成注册KPI的120%，但新用户质量下降了——30日LTV只有25元，CAC是72元。每个新用户净亏47元。建议KPI从『注册数』改为『注册且30日LTV>CAC的新用户数』。」

In [ ]:
# 练习5：漏斗指标博弈检测器
import numpy as np

def gaming_detector(metric_trend, counter_metric_trend, threshold=0.05):
    """检测KPI上升是否伴随反指标恶化"""
    kpi_change = (metric_trend[-1][1] - metric_trend[0][1]) / metric_trend[0][1]
    counter_change = (counter_metric_trend[-1][1] - counter_metric_trend[0][1]) / counter_metric_trend[0][1]
    signals = []
    if kpi_change > threshold and counter_change < -threshold:
        signals.append(f"KPI涨{kpi_change:+.1%}但反指标跌{counter_change:+.1%}——疑似补贴污染/鸡血流量")
    score = 1.0 if signals else 0.0
    return {"risk_score": score, "kpi_change": kpi_change,
            "counter_change": counter_change, "signals": signals,
            "verdict": "HIGH_RISK" if score > 0.5 else "LOW_RISK"}

# 测试：悦选市场团队
registrations = [("W1", 5000), ("W2", 5500), ("W3", 6200), ("W4", 7000)]
retention_7d  = [("W1", 0.22), ("W2", 0.19), ("W3", 0.14), ("W4", 0.08)]
result = gaming_detector(registrations, retention_7d)
print(f"博弈风险: {result["verdict"]}, KPI: {result["kpi_change"]:+.1%}, 反指标: {result["counter_change"]:+.1%}")
for s in result["signals"]:
    print(f"  -> {s}")
print("\n建议：KPI改为「注册且30日LTV>CAC的新用户数」，护栏：7日留存≥15%")

#### 反馈
| 维度 | 内容 |
|------|------|
| ✅ 参考答案 | KPI涨40%同时反指标跌64%——强烈暗示补贴污染。每个新用户净亏47元 |
| ❓ 常见偏差 | 反指标选错——比如用「日活」验证注册质量。注册增加日活自然涨，看不出问题。反指标必须跟KPI逻辑上不自动同步 |
| 🔧 补救 | 学习主课W1D2「好指标5条+护栏指标」——博弈防护是指标设计的核心维度 |

---
### 概念6：HEART框架——Google UX视角的转化衡量 [选修] [替代]

> **关系类型**：[替代]——主课以AARRR为框架，HEART是Google的UX度量体系，提供不同视角
> **来自空白区**：HEART vs AARRR系统对比（永久空白区5.3节）

#### 概念引入

**① 一句话白话**：AARRR是「增长黑客」视角——关注用户量×转化率。HEART是「用户体验」视角——关注用户用得爽不爽。

**② HEART五维度**：
| 维度 | 问什么 | 典型指标 | 与漏斗的关系 |
|------|--------|---------|------------|
| **H**appiness 满意度 | 用户喜欢吗？ | NPS、CSAT | 漏斗不测这个——转化率高≠满意 |
| **E**ngagement 参与度 | 用户深度使用吗？ | 人均Session、核心行为次数 | 漏斗只看到「过了」没看到「用多久」 |
| **A**doption 采纳率 | 新用户采用核心功能吗？ | 首次核心行为完成率 | 漏斗把首次完成作为一个步骤看转化 |
| **R**etention 留存率 | 用户会回来吗？ | 次日/7日/30日留存 | 漏斗最终步骤之后——W2D4会讲 |
| **T**ask Success 任务成功率 | 用户能完成任务吗？ | 任务完成率、时间、错误率 | 漏斗测的就是这个！但HEART强调高效完成而非仅仅完成 |

**核心差异**：AARRR看「用户量的转化效率」，HEART看「单次使用的体验质量」。AARRR告诉你流量在哪卡住了，HEART告诉你卡住的用户在体验什么。

**在沙盘中的用武之地**：悦选加购率从40%提升到43%，但人均加购前犹豫时间从35秒增加到90秒，加购后修改SKU比例从5%增加到18%。AARRR说「好事！」HEART说「详情页信息不清晰，用户需要反复尝试才能确认选择」

**结论**：HEART不替代漏斗——它填补漏斗的盲区：漏斗只看「有没有」，HEART看「好不好」。漏斗是体检报告，HEART是生活质量问卷。

---
### 概念7：漏斗监控看板的工程化设计 [必修] [落地]

> **关系类型**：[落地]——前面所有概念（置信区间、贝叶斯、异常检测、博弈防护）的工程化集成
> **来自空白区**：Traffic Light预警体系 + 前30天数据看板设计（永久空白区1.2节和1.3节）

#### 概念引入

**① 一句话白话**：从「分析师手写周报」到「系统自动监控+告警」——漏斗分析从一次性操作变成持续运行的生产系统。

**② 生产级漏斗监控看板包含五个组件**：数据层(ETL)、计算层(置信区间+贝叶斯+异常检测)、展示层(BI+Traffic Light)、告警层(飞书/钉钉自动推送)、治理层(指标定义+口径记录+SLA)

**Traffic Light预警体系**：绿灯(±1σ)、黄灯(±1~2σ,连续3天升级红灯)、红灯(±2σ外,自动推送+自动分群诊断)

In [ ]:
# 练习7：漏斗监控看板的工程化骨架
import numpy as np

def traffic_light(current_value, baseline_mean, baseline_std):
    """判断当前值属于哪个灯"""
    if baseline_std == 0:
        return "GREEN"
    z = abs(current_value - baseline_mean) / baseline_std
    if z > 2:
        return "RED"
    elif z > 1:
        return "YELLOW"
    else:
        return "GREEN"

# 模拟7天数据
np.random.seed(123)
steps = ["打开->浏览", "浏览->详情", "详情->加购", "加购->下单", "下单->支付"]
baselines = [0.56, 0.70, 0.42, 0.55, 0.67]
stds = [0.02, 0.03, 0.04, 0.03, 0.05]

print("===== 漏斗 Traffic Light 每日状态 =====\n")
print(f"{"步骤":<15} {"基线":>8} {"当前":>8} {"Z-Score":>8} {"灯色":>6}")
print("-" * 55)
for i, (step, base, std) in enumerate(zip(steps, baselines, stds)):
    current = 0.42 if i == 4 else base + np.random.normal(0, std * 0.3)
    z = (current - base) / std
    light = traffic_light(current, base, std)
    emoji = {"GREEN": "G", "YELLOW": "Y", "RED": "R"}[light]
    print(f"{step:<15} {base:>7.1%} {current:>8.1%} {z:>8.2f} {emoji:>6} {light}")

print("\n看板SQL模板 (每日自动运行):")
print("SELECT dt, funnel_step, users, step_conversion,")
print("  CASE WHEN ABS(step_conversion - baseline) > 2*std THEN 'RED'")
print("       WHEN ABS(step_conversion - baseline) > 1*std THEN 'YELLOW'")
print("       ELSE 'GREEN' END AS traffic_light")
print("FROM dws.daily_funnel_agg ...")

# 告警消息模板
print("\n告警推送模板:")
print("""🚨 漏斗异常告警 [CRITICAL] 下单->支付 当前42% 基线67%±5% (-5.0σ)""")
print("""自动分群: 仅Android端异常(iOS正常), 信息流渠道跌至28%(基线45%)""")

#### 反馈
| 维度 | 内容 |
|------|------|
| ✅ 参考答案 | 最后一步触发红灯（-5σ远超2σ阈值）|
| ❓ 常见偏差① | 以为看板就是几张图表——生产级看板需要：ETL+计算+告警+推送+治理，五组件缺一不可 |
| ❓ 常见偏差② | 固定阈值（如<50%一定报警）——应该用滚动基线+标准差。周末效应会产生大量假告警 |
| 🔧 补救 | 给看板加上季节性调整——按周几分组计算基线，避免周末效应误报 |

---
## 第四部分：综合沙盘演练

> 以下4题需要综合运用主课+拓展的多个概念。

---

### 题1：多概念协同——悦选季度漏斗诊断报告

**场景**：悦选CEO要求你在一周内提交季度漏斗诊断报告。数据：近90天每日漏斗数据+渠道/设备/分群可拆分+市场部同时在跑注册送券活动。
**要求**：使用至少4个本拓展概念，给出诊断结论和建议。

In [ ]:
# 综合题1：悦选季度漏斗诊断框架
print("季度漏斗诊断框架")
print("=" * 60)
print("""
Step 1 [概念4 异常检测]: 自动扫描90天x6步骤转化率，标出所有ci_break点
Step 2 [概念3 辛普森]: 确认整体趋势不是由用户结构变化驱动
Step 3 [概念1 置信区间]: 对每个异常点计算CI判断是否统计显著
Step 4 [概念5 博弈]: 检查异常时段是否与注册送券活动重叠
Step 5 [概念2 贝叶斯]: 对小渠道转化率用贝叶斯平滑后对比
Step 6: 综合输出根因+渠道ROI重评估+优化优先级

预期结论:
- 支付转化率下跌且超3sigma -> 排查支付系统变更
- 信息流渠道跌但自然流量正常 -> 流量质量非产品问题
- 注册数涨但留存跌 -> 注册送券引入低质量用户
""")

---
### 题2：沙盘变体——悦选从电商转型为SaaS

**场景**：悦选转型为「设计师SaaS工具」，漏斗变成：访问官网→注册试用→创建工作区→邀请团队成员→首次付费。
**问题**：
1. 转化窗口从7天变成多少？
2. 分群维度哪些保留哪些替换？
3. SaaS场景下异常检测阈值怎么调整？

In [ ]:
# 综合题2：沙盘变体
print("SaaS vs 电商漏斗关键差异")
print("=" * 50)
print("1. 转化窗口: 90天(vs 7天)——决策周期12倍差异")
print("2. 分群维度: 保留渠道来源; 替换设备->公司规模, 城市->行业, 价格敏感度->试用计划")
print("   新增: 角色(创始人/设计师/运营), 邀请成员数(关键激活指标)")
print("3. 阈值调整: 第一步断崖15%(vs 20%), 末步25%(vs 50%)")
print("   新增检测: 邀请成员步骤的断崖(SaaS核心价值检验)")

---
### 题3：边界判断——哪些场景下漏斗分析是错的？

In [ ]:
# 综合题3：漏斗分析的适用边界
scenarios = [
    ("多次往返加购", "不适合", "漏斗假设单向流动。替代: 路径分析(Sankey)或马尔可夫链"),
    ("B2B人机混合销售", "部分适合", "线上漏斗+CRM Pipeline两段式"),
    ("跨端(App+Web+小程序)", "不适合单端", "以User ID为单位的跨端漏斗"),
    ("新功能上线仅50人试用", "谨慎使用", "CI极宽，用贝叶斯结合大盘先验或先做用户录屏"),
]
for s, verdict, reason in scenarios:
    print(f"\n{s}: {verdict}")
    print(f"  {reason}")

---
### 题4：落地——悦选漏斗监控看板完整实现
**任务**：为悦选设计完整的漏斗监控看板——指标定义+ETL+BI+告警+SLA。

In [ ]:
# 综合题4：漏斗监控看板工程方案
print("悦选漏斗监控看板 - 工程方案")
print("=" * 60)
print("\n[1] 指标定义文档: 5个核心指标(FUN-001~005)，含定义、公式、来源、频率、责任人")
print("\n[2] 告警规则: ALT-001倒漏斗(CRITICAL), ALT-002末步暴跌(CRITICAL)")
print("    ALT-003 ci_break(WARNING), ALT-004连续下降(WARNING), ALT-005渠道异常(INFO)")
print("\n[3] BI布局: 第1行KPI栏, 第2行漏斗图+趋势, 第3行分群漏斗, 第4行告警历史")
print("\n[4] SLA: 数据新鲜度T+1 09:00前, CRITICAL告警15min响应, 准确率偏差<0.1%")

---
## 第五部分：知识网络图

### 5.1 概念关系全景表

| 概念 | 关系类型 | 连接对象 | 连接原因 | 协同使用场景 |
|------|---------|---------|---------|------------|
| CI(1) | [前置]-> | 异常检测(4) | CI定义正常范围，ci_break依赖正确CI | 判断异常突破是否统计显著 |
| 贝叶斯(2) | [前置]-> | 异常检测(4) | 小样本用后验代替点估计做基线 | KOL渠道(n<100)用后验均值做基线 |
| 辛普森(3) | [并行]<> | 博弈(5) | 统计结构变化vs人为操纵——都产生整体涨分群跌 | 发现异常时先跑辛普森再跑博弈 |
| 异常检测(4) | [防护]<- | HEART(6) | 异常检测看转化率，HEART补充体验异常 | 转化率正常但NPS暴跌触发HEART检测 |
| 博弈(5) | [防护]-> | 监控看板(7) | 博弈检测逻辑内置于看板告警规则 | 看板自动标记注册涨+留存跌为博弈嫌疑 |
| 监控看板(7) | [落地]<- | CI(1) 异常检测(4) 博弈(5) | 看板是前6个概念的工程化集成 | 生产化漏斗监控系统 |

### 5.2 三版本覆盖对比

| 漏斗分析知识点 | 主课程 | 已有拓展 | 本拓展 |
|-------------|--------|---------|--------|
| 漏斗构建+两视角(流失vs转化) | ✅ 深度 | - | - |
| 漏斗形状诊断(5种) | ✅ 深度 | - | - |
| 漏斗三大局限 | ✅ 深度 | - | - |
| 分群漏斗(新老+渠道+设备) | ✅ 深度 | - | - |
| 转化窗口选择+敏感度 | ✅ 深度 | - | - |
| 漏斗SQL(条件聚合+CTE) | ✅ 中等 | - | - |
| 微转化拆解 | ✅ 深度 | - | - |
| 逆向漏斗分析 | ✅ 中等 | - | - |
| 三角交叉验证(热力图+录屏) | ✅ 深度 | - | - |
| 杠杆效应分析 | ✅ 中等 | - | - |
| **转化率置信区间** | ❌ | ❌ | ✅ 概念1 |
| **贝叶斯漏斗** | ❌ | ❌ | ✅ 概念2 |
| **辛普森悖论形式化** | ❌ | ❌ | ✅ 概念3 |
| **漏斗异常自动检测器** | ❌ | ❌ | ✅ 概念4 |
| **漏斗指标博弈** | ❌ | ❌ | ✅ 概念5 |
| **HEART框架** | ❌ | ❌ | ✅ 概念6 |
| **漏斗监控看板工程** | ❌ | ❌ | ✅ 概念7 |

### 5.3 延伸阅读

| 主题 | 推荐资源 |
|------|---------|
| 贝叶斯数据分析 | Gelman et al. *Bayesian Data Analysis* Ch2 |
| 辛普森悖论 | Judea Pearl *The Book of Why* Ch4 |
| 古德哈特定律 | Muller *Measuring the Measurable* (2018) |
| HEART框架 | Rodden et al. Google Research (2010) |
| 异常检测综述 | Chandola et al. *Anomaly Detection: A Survey* |
| 数据看板工程 | Holistics *The Analytics Setup Guidebook* (2023) |

### 5.4 跨模块连接

| 连接方向 | 关联内容 | 为什么有关 |
|---------|---------|----------|
| W1D2 | 指标三条件/好指标5条 | 博弈防护(概念5)的底层是指标设计——设计好的指标减少博弈空间 |
| W2D2(昨日) | 同期群分析 | 贝叶斯漏斗可应用到同期群留存率的贝叶斯平滑 |
| W2D4(明日) | 留存与LTV | 漏斗最终产出=LTV分子——看板应把转化率和LTV放一起 |
| W5-W6 | SQL分析 | 异常检测(概念4)的窗口函数实现——30天滚动基线 |
| W7-W8 | AB实验 | CI(概念1)是AB实验统计推断的基础 |
| W9 | 因果推断 | 辛普森悖论(概念3)是因果推断入门——理解悖论就理解为什么需要随机化 |
| W10 | 综合案例 | 季度漏斗诊断报告的完整框架——根因分析逻辑W10系统讲 |

---
## 第六部分：元认知自评

### 6.1 学习循环自评表

| # | 检查项 | 是/否 |
|---|--------|------|
| 1 | 我能解释为什么转化率需要置信区间而不只是点估计 | |
| 2 | 我能手算二项分布标准误并解释SE与n的关系 | |
| 3 | 我能用Beta-Binomial在小样本场景下做贝叶斯缩水 | |
| 4 | 我能形式化证明辛普森悖论的数学条件（不只是举例） | |
| 5 | 我能写出FunnelMonitor类并添加新的检测规则 | |
| 6 | 我能列举漏斗指标的6种博弈模式并给出检测方法 | |
| 7 | 我能对比HEART和AARRR在转化分析中的互补关系 | |
| 8 | 我能设计一个完整的漏斗监控看板方案（SQL+告警+SLA） | |
| 9 | 我能判断一个场景是否适用漏斗分析（含边界） | |
| 10 | 我能综合使用4+个本拓展概念完成季度诊断报告 | |

### 6.2 多维度自评量表

| 维度 | 描述 | 自评(0-2) |
|------|------|----------|
| 统计深度 | 能计算/解释漏斗转化率的不确定性（CI+贝叶斯） | |
| 代码能力 | 能写FunnelMonitor/辛普森检测器/贝叶斯更新 | |
| 案例迁移 | 能在电商->SaaS等场景变化中调整漏斗参数和分群 | |
| 博弈意识 | 能识别漏斗指标何时被操纵并提出护栏方案 | |
| 工程思维 | 能从一次性分析升级到生产级监控看板 | |
| 边界判断 | 能判断漏斗分析的适用场景与替代方案 | |

**解读**：总分>=10掌握扎实可以进入W2D4; 6-9有薄弱点需针对性回练; <6建议回主课重新巩固

### 6.3 薄弱点精准强化方案

| 薄弱维度 | 补救行动 |
|---------|---------|
| 统计深度<2 | 重做概念1+2练习，额外：用悦选真实数据计算一周每日每步CI |
| 代码能力<2 | 重做概念4练习，额外：给FunnelMonitor添加渠道间差异检测规则 |
| 案例迁移<2 | 重做综合题2(SaaS变体)，额外：找一个你常用的App设计其漏斗方案 |
| 博弈意识<2 | 重读概念5，额外：找一个「KPI完成但产品变差」的真实案例 |
| 工程思维<2 | 重做概念7+综合题4，额外：写出看板SQL的完整多维度聚合版本 |
| 边界判断<2 | 重做综合题3，额外：对每个不适合场景写出替代方案的具体步骤 |

### 6.4 沙盘复述测试

不看笔记，用自己话回答：
1. 悦选的核心漏斗问题是什么？（转化率停滞+渠道ROI恶化+基础设施薄弱）
2. 你用了哪些方法诊断？（置信区间、贝叶斯、辛普森、异常检测、博弈检测）
3. 你的核心建议是什么？（对CEO/增长VP/市场投放分别怎么说）
4. 如果CEO说明天转型B2B SaaS，你的漏斗方案要改什么？

### 6.5 循环毕业检查清单

- [ ] 主课自诊3题全部答对
- [ ] 7个拓展概念全部完成练习（含代码运行通过）
- [ ] 综合沙盘4题全部完成
- [ ] 能用自己话在3分钟内讲清楚「贝叶斯漏斗为什么比点估计更适合小渠道」
- [ ] 能写出FunnelMonitor类的完整代码（不含参考答案）
- [ ] 能设计一份漏斗监控看板的完整工程方案（含SQL+告警+SLA）
- [ ] 知道漏斗分析的4种不适合场景及其替代方法

---

## 拓展完成标记

```
┌─────────────────────────────────────────────────┐
│  🏁 拓展 W2D3 漏斗分析深度拓展 完成              │
│                                                  │
│  从 点估计 -> 不确定性量化 -> 自动化工程            │
│  从 手写周报 -> 系统监控 -> 博弈防护                │
│                                                  │
│  下一步: W2D4 留存与LTV分析                      │
│  建议: 带着置信区间和贝叶斯思维去学留存            │
│        ——留存率也是转化率的一种（时间维度的转化）   │
└─────────────────────────────────────────────────┘
```

*03analyst_thinking · Week 2 Day 3 拓展 · 基于 analyst-expand v4.1*